In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

def S(phi):
    # Implements the phase matrix S(phi)
    return np.array([[np.exp(1j * phi), 0],
                     [0, np.exp(-1j * phi)]], dtype=complex)

def U(a):
    # Implements the U(a) unitary
    return np.array([[a, np.sqrt(1 - a ** 2)],
                     [np.sqrt(1 - a ** 2), -a]], dtype=complex)

def QSP_matrix(angles, a):
    phi0, phi1, phi2, phi3 = angles
    # Correct order, per problem/Circuit: S(phi0) U(a) S(phi1) U(a) S(phi2) U(a) S(phi3)
    mat = S(phi0) @ U(a) @ S(phi1) @ U(a) @ S(phi2) @ U(a) @ S(phi3)
    return mat

def coefficients(angles):
    """
    Returns the coefficients [alpha, beta] for the cubic+linear QSP polynomial
    corresponding to the given angles as a complex numpy array.
    """
    # Use two linearly independent values of a to set up an exact 2x2 system
    a1 = 1.0
    a2 = 0.5
    A = np.array([
        [a1**3, a1],
        [a2**3, a2]
    ])
    y = np.array([
        QSP_matrix(angles, a1)[0, 0],
        QSP_matrix(angles, a2)[0, 0]
    ])
    alpha, beta = np.linalg.solve(A, y)
    return np.array([alpha, beta])

def run(test_case_input: str) -> str:
    ins = np.array(json.loads(test_case_input))
    coeffs = coefficients(ins)
    outs = [[np.real(elem).item(), np.imag(elem).item()] for elem in coeffs]
    return str(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output, expected_output, rtol = 1e-4)

def test():
    TESTS = [
        ('[0.5,0.8,1.0,1.0]', '[[-0.1707976, -2.4084889], [-0.8166822, 2.2507432]]'),
        ('[-0.20409113, -0.91173829, 0.91173829, 0.20409113]', '[[2.5,0],[-1.5,0]]'),
    ]
    for i, (input_, expected_output) in enumerate(TESTS):
        output = run(input_)
        check(output, expected_output)

test()
